In diesem File findet wieder eine Fusion statt, diesmal werden der Datei "Features_MAIAC_Verkehr_und_Wetter.csv" nun die Daten von sentinel und ERA5 hinzugefügt.

In [1]:
import os
import pandas as pd
import rasterio
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
master_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_MAIAC_Verkehr_und_Wetter.csv"
raster_base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)"

# Ausgabedatei (Die ultimative, vollständige Masterdatei!)
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Master-Datensatz laden
# =========================================================
print("Lade bisherige Masterdatei...")
df_master = pd.read_csv(master_file, sep=";", decimal=",")

# Sicherstellen, dass UTM-Koordinaten als numerische Fließkommazahlen vorliegen
for col in ["x_utm", "y_utm"]:
    if df_master[col].dtype == object:
        df_master[col] = df_master[col].str.replace(",", ".", regex=False)
    df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

# =========================================================
# 3. Raster-Parameter konfigurieren
# =========================================================
# Dictionary für die drei neuen Feature-Ordner
raster_mappings = {
    "S5P_NO2": {"folder": "S5P_NO2", "prefix": "S5P_NO2_"},
    "S5P_UVAI": {"folder": "S5P_UVAI", "prefix": "S5P_UVAI_"},
    "ERA5_BLH": {"folder": "ERA5_BLH", "prefix": "ERA5_BLH_"}
}

# Neue Spalten im DataFrame mit NaN initialisieren
for col in raster_mappings.keys():
    df_master[col] = np.nan

# =========================================================
# 4. Daten aus den GeoTIFFs extrahieren
# =========================================================
print("Extrahiere S5P und ERA5 Satellitendaten...")

unique_dates = df_master["date"].dropna().unique()

for current_date in unique_dates:
    # Zeilen für den aktuellen Tag filtern
    mask = df_master["date"] == current_date
    day_points = df_master.loc[mask, ["x_utm", "y_utm"]].values

    # Wenn keine Koordinaten vorhanden sind, überspringen
    if len(day_points) == 0 or np.isnan(day_points).all():
        continue

    # Als Liste von Tupeln formatieren: [(x1, y1), (x2, y2), ...]
    coords_list = [(x, y) for x, y in day_points]

    # Für jedes der 3 Raster (NO2, UVAI, BLH) die Werte auslesen
    for col_name, mapping in raster_mappings.items():
        tif_filename = f"{mapping['prefix']}{current_date}.tif"
        tif_path = os.path.join(raster_base_dir, mapping["folder"], tif_filename)

        if os.path.exists(tif_path):
            with rasterio.open(tif_path) as src:
                nodata_val = src.nodata

                sampled_values = []
                for val in src.sample(coords_list):
                    v = val[0]

                    # SICHERHEITSFILTER:
                    # Schließe offizielle NoData-Werte oder unnatürliche Extremwerte (> 1 Milliarde) aus
                    if v == nodata_val or v > 1e9 or v < -1e9:
                        sampled_values.append(np.nan)
                    else:
                        sampled_values.append(v)

                # Werte in den Datensatz eintragen
                df_master.loc[mask, col_name] = sampled_values
        else:
            # Nur optionales Feedback, falls du sehen willst, welche Tage Wolkenlücken hatten
            pass

print("Extraktion abgeschlossen.")


### "0" Werte bei S5P NO2 entfernen
# =========================================================
# Ersetze exakte Nullen bei S5P NO2 durch echte Fehlwerte (NaN)
df_master.loc[df_master['S5P_NO2'] == 0.0, 'S5P_NO2'] = np.nan
# =========================================================

# =========================================================
# 5. Speichern des finalen Datensatzes
# =========================================================
# Hilfsspalten x_utm und y_utm können entfernt werden, falls sie stören (optional)
#df_final = df_master.drop(columns=["x_utm", "y_utm"], errors="ignore")

df_master.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"ERFOLG! Dein vollständiger Datensatz für das Machine Learning liegt hier:\n{output_file}")
print("-" * 60)

Lade bisherige Masterdatei...
Extrahiere S5P und ERA5 Satellitendaten...
Extraktion abgeschlossen.
------------------------------------------------------------
ERFOLG! Dein vollständiger Datensatz für das Machine Learning liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_S5P_ERA5_MAIAC_Verkehr_Wetter.csv
------------------------------------------------------------
